In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

In [3]:
DATA_YAML = Path("/workspace/CarDD_YOLO_Reduced/CarDD_YOLO_Reduced/data.yaml")

RUNS_DIR = Path("/workspace/runs/cardd")
EXP4_DIR = RUNS_DIR / "experiment_4_reduced_high_resolution"
DATASET_ROOT = Path("/workspace/CarDD_YOLO_Reduced/CarDD_YOLO_Reduced")

for split in ["train", "val", "test"]:
    image_dir = DATASET_ROOT / "images" / split
    label_dir = DATASET_ROOT / "labels" / split

    print(
        f"{split}: "
        f"{len(list(image_dir.glob('*')))} images | "
        f"{len(list(label_dir.glob('*.txt')))} labels"
    )
DATA_YAML.exists(), EXP4_DIR

train: 1408 images | 1408 labels
val: 810 images | 810 labels
test: 375 images | 374 labels


(True, PosixPath('/workspace/runs/cardd/experiment_4_reduced_high_resolution'))

In [5]:
DATA_YAML = Path("/workspace/CarDD_YOLO_Reduced/CarDD_YOLO_Reduced/data.yaml")

RUNS_DIR = Path("/workspace/runs/cardd")
EXP4_DIR = RUNS_DIR / "experiment_4_reduced_high_resolution"

DATA_YAML.exists(), EXP4_DIR

(True, PosixPath('/workspace/runs/cardd/experiment_4_reduced_high_resolution'))

In [6]:
print(DATA_YAML.read_text())

path: /workspace/CarDD_YOLO_Reduced/CarDD_YOLO_Reduced
train: images/train
val: images/val
test: images/test

names:
  0: dent
  1: scratch
  2: crack
  3: glass shatter
  4: lamp broken
  5: tire flat



In [8]:
model = YOLO("yolo11n.pt")

print("Model loaded successfully.")

Model loaded successfully.


In [9]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        f"GPU memory: "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

PyTorch: 2.8.0+cu128
CUDA available: True
GPU: NVIDIA A40
GPU memory: 44.42 GB


In [10]:
EXP4_CONFIG = {
    "model": "YOLO11n",
    "dataset": "CarDD Reduced",
    "epochs": 100,
    "imgsz": 1280,
    "batch": 8,
    "optimizer": "auto",
    "seed": 42,
    "deterministic": True,
    "amp": True,
    "degrees": 5.0,
    "translate": 0.15,
    "scale": 0.60,
    "fliplr": 0.50,
    "flipud": 0.0,
    "mosaic": 1.0,
    "mixup": 0.10,
    "copy_paste": 0.10,
}

pd.DataFrame([EXP4_CONFIG])

,model,dataset,epochs,imgsz,batch,optimizer,seed,deterministic,amp,degrees,translate,scale,fliplr,flipud,mosaic,mixup,copy_paste
0,YOLO11n,CarDD Reduced,100,1280,8,auto,42,True,True,5.0,0.15,0.6,0.5,0.0,1.0,0.1,0.1


In [11]:
model = YOLO("yolo11n.pt")

results = model.train(
    data=str(DATA_YAML),

    epochs=100,
    imgsz=1280,
    batch=8,

    device=0,
    workers=4,

    optimizer="auto",

    seed=42,
    deterministic=True,

    amp=True,

    degrees=5.0,
    translate=0.15,
    scale=0.60,
    fliplr=0.50,
    flipud=0.0,

    mosaic=1.0,
    mixup=0.10,
    copy_paste=0.10,

    project=str(RUNS_DIR),
    name="experiment_4_reduced_high_resolution",

    exist_ok=True,

    verbose=True
)

New https://pypi.org/project/ultralytics/8.4.152 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.144 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA A40, 45488MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/workspace/CarDD_YOLO_Reduced/CarDD_YOLO_Reduced/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=

In [ ]:
print("Experiment directory:")
print(EXP4_DIR)

print("\nBest model:")
print(EXP4_DIR / "weights" / "best.pt")

print("\nLast model:")
print(EXP4_DIR / "weights" / "last.pt")

In [ ]:
BEST_MODEL = EXP4_DIR / "weights" / "best.pt"

best_model = YOLO(str(BEST_MODEL))

print("Best model loaded:")
print(BEST_MODEL)

In [ ]:
val_results = best_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=1280,
    batch=8,
    device=0,
    workers=4,
    plots=True,
    verbose=True
)

In [ ]:
metrics = val_results.results_dict

overall = {
    "Precision": metrics.get("metrics/precision(B)"),
    "Recall": metrics.get("metrics/recall(B)"),
    "mAP50": metrics.get("metrics/mAP50(B)"),
    "mAP50-95": metrics.get("metrics/mAP50-95(B)")
}

pd.DataFrame([overall])

In [ ]:
class_names = [
    "dent",
    "scratch",
    "crack",
    "glass shatter",
    "lamp broken",
    "tire flat"
]

per_class = []

for i, name in enumerate(class_names):
    per_class.append({
        "Class": name,
        "Precision": val_results.box.p[i],
        "Recall": val_results.box.r[i],
        "mAP50": val_results.box.ap50[i],
        "mAP50-95": val_results.box.ap[i]
    })

per_class_df = pd.DataFrame(per_class)

per_class_df

In [ ]:
class_names = [
    "dent",
    "scratch",
    "crack",
    "glass shatter",
    "lamp broken",
    "tire flat"
]

per_class = []

for i, name in enumerate(class_names):
    per_class.append({
        "Class": name,
        "Precision": val_results.box.p[i],
        "Recall": val_results.box.r[i],
        "mAP50": val_results.box.ap50[i],
        "mAP50-95": val_results.box.ap[i]
    })

per_class_df = pd.DataFrame(per_class)

per_class_df

In [ ]:
small_damage = per_class_df[
    per_class_df["Class"].isin([
        "crack",
        "scratch",
        "dent"
    ])
].copy()

small_damage

In [ ]:
comparison = pd.DataFrame({
    "Metric": [
        "Precision",
        "Recall",
        "mAP50",
        "mAP50-95"
    ],

    "Exp3": [
        0.740389,
        0.608130,
        0.653843,
        0.500476
    ],

    "Exp4": [
        overall["Precision"],
        overall["Recall"],
        overall["mAP50"],
        overall["mAP50-95"]
    ]
})

comparison

In [ ]:
comparison["Change"] = (
    comparison["Exp4"] - comparison["Exp3"]
)

comparison["Change_pp"] = (
    comparison["Change"] * 100
)

comparison

In [ ]:
exp3_classes = pd.DataFrame({
    "Class": [
        "dent",
        "scratch",
        "crack",
        "glass shatter",
        "lamp broken",
        "tire flat"
    ],

    "Exp3_mAP50": [
        0.4875,
        0.4406,
        0.2739,
        0.9870,
        0.8112,
        0.9230
    ],

    "Exp3_mAP50-95": [
        0.2465,
        0.2375,
        0.1286,
        0.8806,
        0.6583,
        0.8513
    ]
})

class_comparison = exp3_classes.merge(
    per_class_df[
        ["Class", "mAP50", "mAP50-95"]
    ],
    on="Class"
)

class_comparison.columns = [
    "Class",
    "Exp3_mAP50",
    "Exp3_mAP50-95",
    "Exp4_mAP50",
    "Exp4_mAP50-95"
]

class_comparison

In [ ]:
class_comparison["mAP50_change"] = (
    class_comparison["Exp4_mAP50"]
    - class_comparison["Exp3_mAP50"]
)

class_comparison["mAP50-95_change"] = (
    class_comparison["Exp4_mAP50-95"]
    - class_comparison["Exp3_mAP50-95"]
)

class_comparison

In [ ]:
plot_df = class_comparison.set_index("Class")

plot_df[
    ["Exp3_mAP50-95", "Exp4_mAP50-95"]
].plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title("Exp3 vs Exp4 — mAP50-95 by Damage Class")
plt.ylabel("mAP50-95")
plt.xlabel("Damage Class")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
comparison_plot = comparison.set_index("Metric")[
    ["Exp3", "Exp4"]
]

comparison_plot.plot(
    kind="bar",
    figsize=(9, 6)
)

plt.title("Exp3 vs Exp4 — Overall Validation Metrics")
plt.ylabel("Score")
plt.xlabel("Metric")
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
RESULTS_CSV = EXP4_DIR / "results.csv"

df = pd.read_csv(RESULTS_CSV)
df.columns = df.columns.str.strip()

df.head()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    df["epoch"],
    df["metrics/mAP50(B)"],
    label="mAP50"
)

plt.plot(
    df["epoch"],
    df["metrics/mAP50-95(B)"],
    label="mAP50-95"
)

plt.xlabel("Epoch")
plt.ylabel("Metric")
plt.title("Exp4 Validation Performance")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 12))

axes[0].plot(
    df["epoch"],
    df["train/box_loss"],
    label="Train Box Loss"
)

axes[0].plot(
    df["epoch"],
    df["val/box_loss"],
    label="Validation Box Loss"
)

axes[0].set_title("Box Loss")
axes[0].legend()
axes[0].grid(True)


axes[1].plot(
    df["epoch"],
    df["train/cls_loss"],
    label="Train Classification Loss"
)

axes[1].plot(
    df["epoch"],
    df["val/cls_loss"],
    label="Validation Classification Loss"
)

axes[1].set_title("Classification Loss")
axes[1].legend()
axes[1].grid(True)


axes[2].plot(
    df["epoch"],
    df["train/dfl_loss"],
    label="Train DFL Loss"
)

axes[2].plot(
    df["epoch"],
    df["val/dfl_loss"],
    label="Validation DFL Loss"
)

axes[2].set_title("DFL Loss")
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
best_idx = df["metrics/mAP50-95(B)"].idxmax()

best_epoch = df.loc[best_idx]

print("Best Epoch:")
print(int(best_epoch["epoch"]))

print("\nBest Metrics:")
print(
    "Precision:",
    best_epoch["metrics/precision(B)"]
)

print(
    "Recall:",
    best_epoch["metrics/recall(B)"]
)

print(
    "mAP50:",
    best_epoch["metrics/mAP50(B)"]
)

print(
    "mAP50-95:",
    best_epoch["metrics/mAP50-95(B)"]
)

In [ ]:
OUTPUT_DIR = RUNS_DIR / "experiment_4_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

comparison.to_csv(
    OUTPUT_DIR / "exp4_vs_exp3_comparison.csv",
    index=False
)

class_comparison.to_csv(
    OUTPUT_DIR / "exp4_vs_exp3_class_comparison.csv",
    index=False
)

print("Saved comparison tables.")

In [ ]:
summary = {
    "Experiment": "Exp4 - Higher Resolution",
    "Model": "YOLO11n",
    "Dataset": "CarDD Reduced",
    "Image Size": 1280,
    "Epochs": 100,
    "Batch": 8,
    "Precision": overall["Precision"],
    "Recall": overall["Recall"],
    "mAP50": overall["mAP50"],
    "mAP50-95": overall["mAP50-95"],
    "Best Epoch": int(best_epoch["epoch"])
}

summary_df = pd.DataFrame([summary])

summary_df.to_csv(
    OUTPUT_DIR / "experiment_4_summary.csv",
    index=False
)

summary_df

In [ ]:
exp3_map = 0.500476
exp4_map = overall["mAP50-95"]

change = exp4_map - exp3_map

print("========== EXP4 CONCLUSION ==========\n")

print(f"Exp3 mAP50-95: {exp3_map:.4f}")
print(f"Exp4 mAP50-95: {exp4_map:.4f}")
print(f"Change: {change:+.4f} ({change*100:+.2f} percentage points)\n")

if change > 0.01:
    print(
        "Higher resolution produced a meaningful improvement "
        "in validation mAP50-95."
    )
elif change > 0:
    print(
        "Higher resolution produced a small improvement "
        "in validation mAP50-95."
    )
else:
    print(
        "Higher resolution did not improve overall validation "
        "mAP50-95."
    )